# Generating Ground Truth Data


In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [3]:
documents = documents_llm
# We'll generate questions only for the LLM Zoomcamp FAQ.

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from dotenv import load_dotenv
load_dotenv()
from rag_helper import RAGBase
from google import genai

google_client = genai.Client()

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


For each document, we:

- convert the document to JSON so we can send it to the LLM
- ask the LLM to return a Questions object
- create one ground truth record for each generated question

In [8]:
import json

user_prompt = json.dumps(doc)

In [ ]:
response = google_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=user_prompt,
    config={
        "system_instruction": data_gen_instructions,
        "response_mime_type": "application/json",
        "response_schema": Questions,
    },
)



In [19]:
result = response.parsed
print(result.questions)

['Is it still possible to enroll in the LLM Zoomcamp course?', 'What are the requirements to receive a certificate for this course?', 'Are there specific deadlines for submitting projects to get the certificate?', 'If I join the course late, can I still qualify for a certificate?', 'Is a project submission necessary to obtain the course certificate?']


In [21]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Is it still possible to enroll in the LLM Zoomcamp course?',
  'document': '74eb249bbf'},
 {'question': 'What are the requirements to receive a certificate for this course?',
  'document': '74eb249bbf'},
 {'question': 'Are there specific deadlines for submitting projects to get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course late, can I still qualify for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Is a project submission necessary to obtain the course certificate?',
  'document': '74eb249bbf'}]

In [9]:
from evaluation_utils import llm_structured, llm_structured_retry

In [10]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        google_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [ ]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

 # This works, but it runs one LLM call after another. 
 # Running it for all documents this way would take too long.   

100%|██████████| 5/5 [00:24<00:00,  4.82s/it]


In [ ]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
# df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

In [12]:
!mkdir -p data
!wget -O data/ground_truth-new.csv https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/data/ground_truth-new.csv

--2026-07-02 08:26:21--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/data/ground_truth-new.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 38613 (38K) [text/plain]
Saving to: ‘data/ground_truth-new.csv’

data/ground_truth-n 100%[===================>]  37.71K  --.-KB/s    in 0.001s  

2026-07-02 08:26:21 (44.2 MB/s) - ‘data/ground_truth-new.csv’ saved [38613/38613]



# Search Evaluation
Now that we have ground truth data, we can evaluate how well our search retrieves the correct documents.

For each question in our ground truth dataset, we run search. Then we check whether the results include the correct document.

In [13]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [14]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [15]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [ ]:
# Start with one ground truth record:
q = ground_truth[0]
q

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [ ]:
# Run search for this question:
doc_id = q["document"]
results = text_search(query=q["question"])

In [18]:
# compare the retrieved document IDs with the correct document ID:

for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
0fab61eca2 == 74eb249bbf: False
610ccb23c0 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
acf8fa5356 == 74eb249bbf: False


In [ ]:
# Then turn this comparison into a relevance list. In this lesson, relevance means whether a retrieved document is the correct document for this question.

relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance
# 1 means the retrieved document has the same ID as the correct document.

[1, 0, 0, 0, 0]

In [22]:
# Put this logic into a function:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]



Is it okay to join the course late if I just found it now?


[1, 0, 0, 0, 0]

In [23]:
# Now do the same thing for all ground truth questions:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [24]:
# Call it for the first 15 ground truth questions:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

# Look at the results:

relevance_total_text

100%|██████████| 15/15 [00:00<00:00, 295.30it/s]


[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

Next, make the relevance functions generic. We start with text search, but later we may want to evaluate vector search, hybrid search, or another retrieval method. The relevance logic is the same. Only the search function changes.

In [25]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [26]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [27]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

100%|██████████| 15/15 [00:00<00:00, 62.52it/s]


[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [28]:
# Now run it for all ground truth questions:


relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

100%|██████████| 15/15 [00:00<00:00, 90.07it/s]


[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

# Hit Rate
Hit Rate (also called Recall@k) measures the fraction of queries where the correct document appears anywhere in the results:



In [31]:
cnt = 0

for line in relevance_total:
    if 1 in line:
        cnt = cnt + 1

cnt
cnt / len(relevance_total)
# 0.933

0.8666666666666667

In [32]:
# Put the same logic into a function:

def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

# Check it on the same example:

hit_rate(relevance_total)

0.8666666666666667

# Mean Reciprocal Rank (MRR)
Hit Rate tells us if we found the right document, but not where it was.

MRR also considers the position.

For each query, the score is based on the rank of the first correct document:

position 1: score is 1.0

position 2: score is 0.5

position 3: score is 0.333

not found: score is 0

In [33]:
total_score = 0.0

for line in relevance_total:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

10.666666666666666

In [35]:
# Divide it by the number of queries:

total_score / len(relevance_total)

0.711111111111111

MRR is the average of these scores across all queries. It rewards systems that put the correct document near the top.

In [36]:
# Put the same logic into a function:

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)
# Check it on the same example:

mrr(relevance_total)

0.711111111111111

In [37]:
# Putting it together
# Wrap the metrics in a reusable evaluation function:

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [38]:
evaluate(
    ground_truth,
    text_search
)


100%|██████████| 395/395 [00:01<00:00, 300.47it/s]


{'hit_rate': 0.660759493670886, 'mrr': 0.5805485232067511}

# Search Parameter Tuning
Video: Watch this lesson

In the previous lesson, we defined Hit Rate, MRR, and the evaluate function. Now we can use them to tune search parameters.

Instead of guessing which settings are better, we measure them on the ground truth dataset.

So far we've boosted question to 3.0. The idea was that a query should match the FAQ question. That kind of match should count for more than matching the answer text. It sounds reasonable. But it's a guess, and now we can check it against data instead of trusting it.

This is the main benefit of offline evaluation. We change one parameter, run the same questions again, and see whether the metric moves. The dataset stays fixed, so the comparison is fair.

In [39]:
# Trying different boosts
# Start with a search function where the question boost is configurable:

def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [40]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

100%|██████████| 395/395 [00:03<00:00, 112.11it/s]


boost=0.5: {'hit_rate': 0.6962025316455697, 'mrr': 0.5930379746835442}


100%|██████████| 395/395 [00:01<00:00, 253.08it/s]


boost=1.0: {'hit_rate': 0.6962025316455697, 'mrr': 0.6004641350210971}


100%|██████████| 395/395 [00:01<00:00, 266.04it/s]


boost=3.0: {'hit_rate': 0.660759493670886, 'mrr': 0.5805485232067511}


100%|██████████| 395/395 [00:01<00:00, 291.48it/s]


boost=5.0: {'hit_rate': 0.6455696202531646, 'mrr': 0.5588607594936709}


100%|██████████| 395/395 [00:01<00:00, 279.09it/s]

boost=10.0: {'hit_rate': 0.6379746835443038, 'mrr': 0.5434177215189874}


Increasing the question boost makes the metrics worse, not better. The best value here is 1.0, no boost at all. That's already the opposite of what the intuition predicted.

But this is only one parameter. We can also tune answer and section together with question.

Define a search function with all three boosts:


In [41]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [42]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


100%|██████████| 395/395 [00:01<00:00, 251.01it/s]


Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


100%|██████████| 395/395 [00:01<00:00, 277.58it/s]


Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

100%|██████████| 395/395 [00:01<00:00, 277.88it/s]


Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


100%|██████████| 395/395 [00:01<00:00, 277.71it/s]


Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


100%|██████████| 395/395 [00:01<00:00, 301.29it/s]


In [43]:
# Sort by MRR:

df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)


,question,answer,section,hit_rate,mrr
18,2.0,4.0,0.1,0.756962,0.651603
34,5.0,10.0,0.2,0.756962,0.651519
3,1.0,2.0,0.1,0.754430,0.650464
35,5.0,10.0,0.5,0.754430,0.650464
19,2.0,4.0,0.2,0.754430,0.650464
33,5.0,10.0,0.1,0.751899,0.649241
4,1.0,2.0,0.2,0.751899,0.648650
20,2.0,4.0,0.5,0.746835,0.644895
7,1.0,4.0,0.2,0.751899,0.636160
6,1.0,4.0,0.1,0.751899,0.635823


In [44]:
# Define the search function with these boosts:

def text_search(query):
    boost_dict = {
        "question": 2.0,
        "answer": 4.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

# Tuning Workflow
Search parameters can look arbitrary. This includes field boosts, number of results, filters, and other settings. Evaluation gives us a way to compare settings with evidence.

Grid search is fine when there are only a few settings. For a larger parameter space, use a smarter search strategy. You can sample random combinations, use Bayesian optimization, or keep a validation split so you don't overfit the evaluation set.

For text search on our dataset, grid search takes about one second per combination. That makes it practical to try many options. When each evaluation takes minutes instead of seconds, grid search becomes too expensive. In those cases, use Bayesian optimization with a library like hyperopt. It explores the parameter space more efficiently by focusing on combinations that are likely to improve the metric.



# Top-K tradeoffs
We return 5 results from search. Increasing top-K to 10 would improve hit rate because there are more chances to find the correct document. But more results means more context sent to the LLM. That costs more and makes it harder for the model to identify what is relevant. Five results is a reasonable default for short FAQ-style document

# RAG and Agent Evaluation
So far, we evaluated retrieval. We checked whether search returns the document that should answer the question.

That is only the first step. A complete application still needs to produce a final answer. For RAG, this means checking the generated answer. For agents, it also means looking at the tool calls the model made before producing the answer.

RAG evaluation checks the whole flow together.

This includes:

search
prompt
LLM
If the final answer is bad, the problem can come from any of these steps. The search might retrieve the wrong document, the prompt might omit important context, or the LLM might ignore the context.

# Generating RAG Answers


In the first part of this module, we evaluated search quality. We checked whether the right document appeared in the search results.

Now we evaluate the full RAG pipeline. For each generated question, we run RAG and save the answer produced by the LLM. Later, we'll compare this answer with the original FAQ answer.

This is the A->Q->A' setup:

- A = original answer in the FAQ
- Q = generated question from this answer
- A' = answer produced by our RAG system

If A' is close to A, the RAG system is doing a good job.

In [1]:
# Load the ground truth questions:

import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
# Load the FAQ documents and the search index:

from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
print(documents[0])

{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}


In [6]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc


print(doc_idx)    

{'74eb249bbf': {'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, '977bf7786c': {'id': '977bf7786c', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?', 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}, '489dd1c9d9': {'id': '489dd1c9d9', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessi

In [ ]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=google_client,
)

In [ ]:
# For each question, RAGBase searches the FAQ, builds a prompt with the retrieved context, and asks the LLM to answer. 
# We save the answer so the next lesson can judge it.

# Run RAG for one question:

rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

# Check the cost of this call:

assistant.total_cost()


# Get the original answer from the document ID:

doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

In [ ]:
# Now save both answers in one record:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

In [ ]:
# Processing all questions
# Create a function that processes one ground truth record:

def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

# Test it on one record:

answer_record = generate_rag_answer(ground_truth[0])
answer_record

In [9]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
!mkdir -p data
!wget -O data/rag-answers-new.csv {PREFIX}/04-evaluation/data/rag-answers-new.csv

--2026-07-02 11:13:51--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/data/rag-answers-new.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 318108 (311K) [text/plain]
Saving to: ‘data/rag-answers-new.csv’

data/rag-answers-ne 100%[===================>] 310.65K  --.-KB/s    in 0.002s  

2026-07-02 11:13:52 (124 MB/s) - ‘data/rag-answers-new.csv’ saved [318108/318108]

